In [ ]:
!pip install xgboost


  Using cached xgboost-3.1.1-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nccl_cu12-2.28.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
Using cached xgboost-3.1.1-py3-none-manylinux_2_28_x86_64.whl (115.9 MB)
   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/296.8 MB 237.7 kB/s eta 0:19:21

In [5]:
# ============================================
# 1. Import libraries
# ============================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# ============================================
# 2. Load Data
# ============================================
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

# ============================================
# 3. FEATURE ENGINEERING
# ============================================

# ----- Convert Sex -----
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})

# ----- Fill Missing Age -----
train["Age"].fillna(train["Age"].median(), inplace=True)
test["Age"].fillna(test["Age"].median(), inplace=True)

# ----- Fill Missing Embarked -----
train["Embarked"].fillna(train["Embarked"].mode()[0], inplace=True)
test["Embarked"].fillna(test["Embarked"].mode()[0], inplace=True)

# ----- Fill Missing Fare -----
test["Fare"].fillna(test["Fare"].median(), inplace=True)

# =================================================
# TITLE EXTRACTION (Huge boost)
# =================================================

train["Title"] = train["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Replace rare titles
for df in [train, test]:
    df["Title"] = df["Title"].replace(
        ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", 
         "Sir", "Jonkheer", "Dona"], "Rare"
    )
    df["Title"] = df["Title"].replace(["Mlle", "Ms"], "Miss")
    df["Title"] = df["Title"].replace("Mme", "Mrs")

# One-hot encode Title
train = pd.get_dummies(train, columns=["Title"])
test = pd.get_dummies(test, columns=["Title"])

# =================================================
# FAMILY FEATURES (Very important)
# =================================================
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

# =================================================
# AGE BINNING (Improves ML models)
# =================================================
train["AgeBin"] = pd.cut(train["Age"], bins=[0, 12, 18, 35, 60, 100], labels=[0, 1, 2, 3, 4])
test["AgeBin"] = pd.cut(test["Age"], bins=[0, 12, 18, 35, 60, 100], labels=[0, 1, 2, 3, 4])

# =================================================
# FARE BINNING
# =================================================
train["FareBin"] = pd.qcut(train["Fare"], 4, labels=[0, 1, 2, 3])
test["FareBin"] = pd.qcut(test["Fare"], 4, labels=[0, 1, 2, 3])

# =================================================
# ONE-HOT ENCODE EMBARKED
# =================================================
train = pd.get_dummies(train, columns=['Embarked'])
test = pd.get_dummies(test, columns=['Embarked'])

# =================================================
# ALIGN COLUMNS
# =================================================
train, test = train.align(test, join="left", axis=1, fill_value=0)

# =================================================
# FEATURE SELECTION
# =================================================

target = "Survived"

features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "AgeBin", "FareBin",
    "Embarked_C", "Embarked_Q", "Embarked_S",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare"
]

X = train[features]
y = train[target]
X_test = test[features]

# ============================================
# TRAIN/VALIDATION SPLIT
# ============================================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================
# 4. RANDOM FOREST MODEL
# ============================================

rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=12,
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_valid)

rf_acc = accuracy_score(y_valid, rf_pred)
print("✅ Random Forest Accuracy:", rf_acc)


# ============================================
# 5. XGBOOST MODEL (Best Kaggle Score)
# ============================================

xgb = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_valid)

xgb_acc = accuracy_score(y_valid, xgb_pred)
print("✅ XGBoost Accuracy:", xgb_acc)

# ============================================
# 6. SELECT BEST MODEL
# ============================================

best_model = xgb if xgb_acc > rf_acc else rf_model

# ============================================
# 7. FINAL PREDICTIONS FOR KAGGLE
# ============================================

final_predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_predictions
})

submission.to_csv("submission_optimized.csv", index=False)
print("✅ submission_optimized.csv saved!")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/titanic/train.csv'